In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/HeyCareLog_Dataset'

import os
os.makedirs(f'{BASE}/models/disfluency', exist_ok=True)
os.makedirs(f'{BASE}/results', exist_ok=True)

print('Drive connected!')
print(f'BASE = {BASE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive connected!
BASE = /content/drive/MyDrive/HeyCareLog_Dataset


Install Libraries

In [ ]:
# Install libraries for disfluency removal
# Using only lightweight models to avoid high RAM usage

!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q evaluate
!pip install -q sentencepiece
!pip install -q rouge_score

import torch
print('Libraries installed!')
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')

Libraries installed!
GPU available: True
GPU name: Tesla T4


Load My Data

In [ ]:
# ------------------------------------------------------------
# CELL 3 - Loading Dataset (Train / Validation / Test)
# ------------------------------------------------------------
# This cell loads the dataset splits required for training the model:
#   - Training set: used to train the model
#   - Validation set: used to tune and evaluate during training
#   - Test set: used for final evaluation of model performance
#
# INPUT:
#   speech_to_text_output  → Noisy ASR output containing disfluencies
#
# TARGET:
#   expected_cleaned_text  → Clean, corrected version of the text
#
# ------------------------------------------------------------
# DATA CHARACTERISTICS (Disfluency Types)
# The dataset contains 4 types of speech disfluencies:
#
# 1. filler_word_removal only
# 2. filler + self_correction_handling
# 3. filler + self_correction + repetition
# 4. filler + repetition
#
# This distribution shows that most samples include both
# filler words and self-corrections, making the dataset
# realistic for disfluency removal tasks.
# ------------------------------------------------------------


import pandas as pd

train_df = pd.read_csv(f'{BASE}/labels/train_text.csv')
val_df   = pd.read_csv(f'{BASE}/labels/val_text.csv')
test_df  = pd.read_csv(f'{BASE}/labels/test_text.csv')

print(f'Train: {len(train_df)} rows')
print(f'Val:   {len(val_df)} rows')
print(f'Test:  {len(test_df)} rows')

# show disfluency type distribution
print('\nDisfluency types in training data:')
print(train_df['auto_edit_features'].value_counts())

# show one real example of each type
print('\n=== EXAMPLE: Filler only ===')
ex1 = train_df[train_df['auto_edit_features'] == 'filler_word_removal'].iloc[0]
print(f'NOISY:  {str(ex1["speech_to_text_output"])[:150]}')
print(f'CLEAN:  {str(ex1["expected_cleaned_text"])[:150]}')

print('\n=== EXAMPLE: Filler + Self-Correction ===')
ex2 = train_df[train_df['auto_edit_features'].str.contains('self_correction')].iloc[3]
print(f'NOISY:  {str(ex2["speech_to_text_output"])[80:280]}')
print(f'CLEAN:  {str(ex2["expected_cleaned_text"])[80:260]}')

Train: 967 rows
Val:   121 rows
Test:  121 rows

Disfluency types in training data:
auto_edit_features
filler_word_removal; spoken_self_correction_handling                        540
filler_word_removal                                                         280
filler_word_removal; spoken_self_correction_handling; repetition_removal     94
filler_word_removal; repetition_removal                                      53
Name: count, dtype: int64

=== EXAMPLE: Filler only ===
NOISY:  Today is March 10 2026 this is for patient P024 in the male branch Morning care was done at 7:38 AM He had uh full bath with warm water and towel and 
CLEAN:  Today is 2026-03-10. This log is for patient P024 in the male branch. Morning personal care was completed at 7:38 AM. He had a full bath with warm wat

=== EXAMPLE: Filler + Self-Correction ===
NOISY:   was done at 7:39 AM She had uh bed bath with warm water and towel and body cream was applied Breakfast was given at 10:14 AM She had breakfast pittu wi

Load Evaluation Metrics

In [ ]:
# Load BLEU and ROUGE metrics
#
# BLEU Score:  measures word n-gram overlap between prediction and reference
#              Higher = better. Target > 0.60 for this task.
#
# ROUGE-L:     measures longest common subsequence similarity
#              Higher = better. Target > 0.75 for this task.
#
# These are standard metrics used in text generation research papers.
# Lewis et al. (2020) BART paper uses BLEU and ROUGE for text correction evaluation.

import evaluate

bleu_metric  = evaluate.load('bleu')
rouge_metric = evaluate.load('rouge')

def compute_scores(predictions, references):
    bleu  = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references]
    )['bleu']

    rouge = rouge_metric.compute(
        predictions=predictions,
        references=references
    )['rougeL']

    return {
        'BLEU':   round(bleu,  4),
        'ROUGE-L':round(rouge, 4)
    }

print('Evaluation metrics loaded!')
print('BLEU  — measures word overlap (higher = better)')
print('ROUGE-L — measures sentence similarity (higher = better)')

Evaluation metrics loaded!
BLEU  — measures word overlap (higher = better)
ROUGE-L — measures sentence similarity (higher = better)


Zero-Shot Baseline Evaluation

In [ ]:
# Zero-shot baseline for T5-small and BART-base
# Zero-shot = no training, run model as-is
# This shows baseline performance before fine-tuning
# and proves fine-tuning is necessary for caregiver speech

import torch, gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def zero_shot_eval(model_name, prefix='', n=40):
    print(f'  Loading {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    preds = []
    refs  = []

    for _, row in test_df.head(n).iterrows():
        inp  = prefix + str(row['speech_to_text_output'])
        toks = tokenizer(inp, return_tensors='pt',
                         max_length=512, truncation=True)
        with torch.no_grad():
            out = model.generate(**toks, max_new_tokens=400,
                                 num_beams=4, early_stopping=True)
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(str(row['expected_cleaned_text']))

    scores = compute_scores(preds, refs)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return scores

zero_shot_results = {}

print('[1/2] T5-small zero-shot...')
zero_shot_results['T5-small (zero-shot)'] = zero_shot_eval(
    't5-small', prefix='clean text: ')
print(f'  {zero_shot_results["T5-small (zero-shot)"]}')

print('\n[2/2] BART-base zero-shot...')
zero_shot_results['BART-base (zero-shot)'] = zero_shot_eval(
    'facebook/bart-base', prefix='')
print(f'  {zero_shot_results["BART-base (zero-shot)"]}')

print('\n=== ZERO-SHOT BASELINES ===')
for name, scores in zero_shot_results.items():
    print(f'{name}: BLEU={scores["BLEU"]}  ROUGE-L={scores["ROUGE-L"]}')

[1/2] T5-small zero-shot...
  Loading t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  {'BLEU': 0.0, 'ROUGE-L': np.float64(0.0)}

[2/2] BART-base zero-shot...
  Loading facebook/bart-base...


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

  {'BLEU': 0.3491, 'ROUGE-L': np.float64(0.7705)}

=== ZERO-SHOT BASELINES ===
T5-small (zero-shot): BLEU=0.0  ROUGE-L=0.0
BART-base (zero-shot): BLEU=0.3491  ROUGE-L=0.7705


Fine-Tune BART-base

In [ ]:
# Fine-tune BART-base

from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import gc, torch

MODEL_NAME = 'facebook/bart-base'

print(f'Loading {MODEL_NAME}...')
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model     = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

MAX_INPUT  = 512
MAX_TARGET = 400

def tokenize_pair(batch):
    model_inputs = tokenizer(
        batch['noisy'],
        max_length=MAX_INPUT,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        batch['clean'],
        max_length=MAX_TARGET,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# prepare datasets
train_data = Dataset.from_dict({
    'noisy': train_df['speech_to_text_output'].tolist(),
    'clean': train_df['expected_cleaned_text'].tolist()
}).map(tokenize_pair, batched=True, batch_size=16)

val_data = Dataset.from_dict({
    'noisy': val_df['speech_to_text_output'].tolist(),
    'clean': val_df['expected_cleaned_text'].tolist()
}).map(tokenize_pair, batched=True, batch_size=16)

print(f'Training examples: {len(train_data)}')
print(f'Validation examples: {len(val_data)}')

training_args = Seq2SeqTrainingArguments(
    output_dir=f'{BASE}/models/disfluency/bart',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=400,
    fp16=True,
    logging_steps=50,
    report_to=['none'],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True
    ),
)

print('\nStarting BART fine-tuning...')
print('Expected time: 20-40 minutes on T4 GPU')
print('Watch eval_loss decrease each epoch — model is learning')
print()

trainer.train()

print('\nFine-tuning complete!')
print(f'Model saved to: {BASE}/models/disfluency/bart')

Loading facebook/bart-base...


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/967 [00:00<?, ? examples/s]

Map:   0%|          | 0/121 [00:00<?, ? examples/s]

Training examples: 967
Validation examples: 121

Starting BART fine-tuning...
Expected time: 20-40 minutes on T4 GPU
Watch eval_loss decrease each epoch — model is learning



...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Fine-tuning complete!
Model saved to: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart


Fine-tune T5-small

In [ ]:
# Fine-tune T5-small

from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import gc, torch

T5_NAME = 't5-small'

print(f'Loading {T5_NAME}...')
t5_tok   = T5Tokenizer.from_pretrained(T5_NAME)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_NAME)

def tokenize_t5(batch):
    inputs = ['clean caregiver speech: ' + t for t in batch['noisy']]
    model_inputs = t5_tok(
        inputs, max_length=512, truncation=True, padding='max_length'
    )
    labels = t5_tok(
        batch['clean'], max_length=400, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

t5_train = Dataset.from_dict({
    'noisy': train_df['speech_to_text_output'].tolist(),
    'clean': train_df['expected_cleaned_text'].tolist()
}).map(tokenize_t5, batched=True, batch_size=16)

t5_val = Dataset.from_dict({
    'noisy': val_df['speech_to_text_output'].tolist(),
    'clean': val_df['expected_cleaned_text'].tolist()
}).map(tokenize_t5, batched=True, batch_size=16)

t5_args = Seq2SeqTrainingArguments(
    output_dir=f'{BASE}/models/disfluency/t5small',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=400,
    fp16=True,
    logging_steps=50,
    report_to=['none'],
)

t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_args,
    train_dataset=t5_train,
    eval_dataset=t5_val,
    processing_class=t5_tok,
    data_collator=DataCollatorForSeq2Seq(t5_tok, model=t5_model),
)

print('\nStarting T5-small fine-tuning...')
print('Expected time: 15-25 minutes on T4 GPU')
print()

t5_trainer.train()

print('\nT5-small fine-tuning complete!')
print(f'Model saved to: {BASE}/models/disfluency/t5small')

del t5_model, t5_tok, t5_trainer
gc.collect()
torch.cuda.empty_cache()

Loading t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/967 [00:00<?, ? examples/s]

Map:   0%|          | 0/121 [00:00<?, ? examples/s]


Starting T5-small fine-tuning...
Expected time: 15-25 minutes on T4 GPU



...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



T5-small fine-tuning complete!
Model saved to: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/t5small


In [ ]:
# find where BART model was saved
import os

base_path = f'{BASE}/models/disfluency/bart'
print('Contents of bart folder:')
for item in os.listdir(base_path):
    full_path = os.path.join(base_path, item)
    if os.path.isdir(full_path):
        print(f'  FOLDER: {item}')
        print(f'    Contents: {os.listdir(full_path)}')
    else:
        print(f'  FILE: {item}')

Contents of bart folder:
  FOLDER: checkpoint-121
    Contents: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']
  FOLDER: checkpoint-242
    Contents: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']
  FOLDER: checkpoint-363
    Contents: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


In [ ]:
import os
t5_base = f'{BASE}/models/disfluency/t5small'
print('T5 checkpoints:')
for item in os.listdir(t5_base):
    print(f'  {item}')

T5 checkpoints:
  checkpoint-121
  checkpoint-242
  checkpoint-363


 Evaluate all 4 models

In [ ]:
# Evaluate all 4 models

import torch, gc
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    T5Tokenizer, T5ForConditionalGeneration
)

# correct checkpoint paths for both models
BART_PATH = f'{BASE}/models/disfluency/bart/checkpoint-363'
T5_PATH   = f'{BASE}/models/disfluency/t5small/checkpoint-363'

print(f'BART path: {BART_PATH}')
print(f'T5 path:   {T5_PATH}')

def evaluate_finetuned(model_obj, tok_obj, prefix=''):
    model_obj.eval()
    preds = []
    refs  = []

    for _, row in test_df.iterrows():
        inp  = prefix + str(row['speech_to_text_output'])
        toks = tok_obj(inp, return_tensors='pt',
                       max_length=512, truncation=True)
        with torch.no_grad():
            out = model_obj.generate(
                **toks, max_new_tokens=400,
                num_beams=4, early_stopping=True
            )
        pred = tok_obj.decode(out[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(str(row['expected_cleaned_text']))

    return compute_scores(preds, refs)

# zero-shot results from Cell 5
all_results = {
    'T5-small (zero-shot)':  {'BLEU': 0.0,    'ROUGE-L': 0.0},
    'BART-base (zero-shot)': {'BLEU': 0.3491, 'ROUGE-L': 0.7705},
}

# evaluate fine-tuned BART
print('\nEvaluating fine-tuned BART-base...')
bart_tok   = BartTokenizer.from_pretrained(BART_PATH)
bart_model = BartForConditionalGeneration.from_pretrained(BART_PATH)
all_results['BART-base (fine-tuned)'] = evaluate_finetuned(
    bart_model, bart_tok, prefix='')
print(f'  BART fine-tuned: {all_results["BART-base (fine-tuned)"]}')

del bart_model, bart_tok
gc.collect()
torch.cuda.empty_cache()

# evaluate fine-tuned T5
print('\nEvaluating fine-tuned T5-small...')
t5_tok   = T5Tokenizer.from_pretrained(T5_PATH)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_PATH)
all_results['T5-small (fine-tuned)'] = evaluate_finetuned(
    t5_model, t5_tok,
    prefix='clean caregiver speech: ')
print(f'  T5 fine-tuned: {all_results["T5-small (fine-tuned)"]}')

del t5_model, t5_tok
gc.collect()
torch.cuda.empty_cache()

print('\nAll models evaluated!')

BART path: /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/bart/checkpoint-363
T5 path:   /content/drive/MyDrive/HeyCareLog_Dataset/models/disfluency/t5small/checkpoint-363

Evaluating fine-tuned BART-base...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

  BART fine-tuned: {'BLEU': 0.843, 'ROUGE-L': np.float64(0.9227)}

Evaluating fine-tuned T5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

  T5 fine-tuned: {'BLEU': 0.9872, 'ROUGE-L': np.float64(0.9937)}

All models evaluated!


In [ ]:
# Check if T5 is overfitting or genuinely learned
# Test on completely new sentences NOT in my dataset

from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

T5_PATH = f'{BASE}/models/disfluency/t5small/checkpoint-363'
t5_tok   = T5Tokenizer.from_pretrained(T5_PATH)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_PATH)
t5_model.eval()

def clean_text(noisy):
    inp  = 'clean caregiver speech: ' + noisy
    toks = t5_tok(inp, return_tensors='pt',
                  max_length=512, truncation=True)
    with torch.no_grad():
        out = t5_model.generate(**toks, max_new_tokens=400,
                                num_beams=4)
    return t5_tok.decode(out[0], skip_special_tokens=True)

print('=== OVERFITTING CHECK ===')
print('Testing on NEW sentences never seen during training')
print()

# these are completely new sentences not in my dataset
new_sentences = [
    # new filler pattern
    "She had uh uh rice for lunch and drank um about 300 ml water",

    # new self-correction
    "Medicine was given after breakfast uh one tablet no sorry two tablets as the doctor said",

    # new repetition
    "Diaper was changed diaper was changed 2 times today mood was calm",

    # new complex example
    "She had uh full body bath this morning and um breakfast was given she ate half of the rice and drank about 150 ml water medicine after dinner was refused no sorry was given",

    # completely different structure
    "Patient had uh physiotherapy session today um she walked with support for about 10 minutes no sorry 15 minutes",
]

for i, sentence in enumerate(new_sentences, 1):
    result = clean_text(sentence)
    print(f'Test {i}:')
    print(f'  INPUT:  {sentence}')
    print(f'  OUTPUT: {result}')
    print()

import gc
del t5_model, t5_tok
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

=== OVERFITTING CHECK ===
Testing on NEW sentences never seen during training

Test 1:
  INPUT:  She had uh uh rice for lunch and drank um about 300 ml water
  OUTPUT: 

Test 2:
  INPUT:  Medicine was given after breakfast uh one tablet no sorry two tablets as the doctor said
  OUTPUT: Medicine was given after breakfast uh one tablet no sorry two tablets as instructed by the doctor.

Test 3:
  INPUT:  Diaper was changed diaper was changed 2 times today mood was calm
  OUTPUT: 

Test 4:
  INPUT:  She had uh full body bath this morning and um breakfast was given she ate half of the rice and drank about 150 ml water medicine after dinner was refused no sorry was given
  OUTPUT: 

Test 5:
  INPUT:  Patient had uh physiotherapy session today um she walked with support for about 10 minutes no sorry 15 minutes
  OUTPUT: 



In [ ]:
# Check if BART fine-tuned model is overfitting or genuinely learned
# Test on completely new sentences NOT in my dataset

from transformers import BartTokenizer, BartForConditionalGeneration
import torch

BART_PATH = f'{BASE}/models/disfluency/bart/checkpoint-363'
bart_tok   = BartTokenizer.from_pretrained(BART_PATH)
bart_model = BartForConditionalGeneration.from_pretrained(BART_PATH)
bart_model.eval()

def clean_bart(noisy):
    toks = bart_tok(noisy, return_tensors='pt',
                    max_length=512, truncation=True)
    with torch.no_grad():
        out = bart_model.generate(**toks, max_new_tokens=400,
                                  num_beams=4)
    return bart_tok.decode(out[0], skip_special_tokens=True)

print('=== BART OVERFITTING CHECK ===')
print('Testing on NEW sentences never seen during training')
print()

new_sentences = [
    # new filler pattern
    "She had uh uh rice for lunch and drank um about 300 ml water",

    # new self-correction
    "Medicine was given after breakfast uh one tablet no sorry two tablets as the doctor said",

    # new repetitiona
    "Diaper was changed diaper was changed 2 times today mood was calm",

    # complex new example
    "She had uh full body bath this morning and um breakfast was given she ate half of the rice and drank about 150 ml water medicine after dinner was refused no sorry was given",

    # completely different structure
    "Patient had uh physiotherapy session today um she walked with support for about 10 minutes no sorry 15 minutes",
]

for i, sentence in enumerate(new_sentences, 1):
    result = clean_bart(sentence)
    print(f'Test {i}:')
    print(f'  INPUT:  {sentence}')
    print(f'  OUTPUT: {result}')
    print()

import gc
del bart_model, bart_tok
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

=== BART OVERFITTING CHECK ===
Testing on NEW sentences never seen during training

Test 1:
  INPUT:  She had uh uh rice for lunch and drank um about 300 ml water
  OUTPUT: She had uh uh rice for lunch and drank um about 300 ml of water.

Test 2:
  INPUT:  Medicine was given after breakfast uh one tablet no sorry two tablets as the doctor said
  OUTPUT: Medicine was given after breakfast, one tablet, two tablets as instructed by the doctor.

Test 3:
  INPUT:  Diaper was changed diaper was changed 2 times today mood was calm
  OUTPUT: Diaper was changed diaper was changed 2 times today. Mood was calm.

Test 4:
  INPUT:  She had uh full body bath this morning and um breakfast was given she ate half of the rice and drank about 150 ml water medicine after dinner was refused no sorry was given
  OUTPUT: She had a full body bath this morning and um breakfast was given she ate half of the rice and drank about 150 ml of water. Medicine after dinner was refused. No sorry was given.

Test 5:
  I